<a href="https://colab.research.google.com/github/rcs227/hyrule_warriors/blob/main/raw_data/Librosa/test/librosa_hum_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import scipy.signal as signal
import librosa
import librosa.display
import matplotlib.pyplot as plt

def midi_to_hz(midi_note):
    """Convert MIDI note number to Hz."""
    return 440.0 * (2.0 ** ((midi_note - 69) / 12.0))

def generate_synthetic_hum(
    midi_notes,
    note_durations,
    sr=22050,
    vibrato_rate=5.0,
    vibrato_depth=2.0,
    cutoff_freq=400.0
):
    """
    Generates a synthetic hummed audio signal with vibrato, continuous
    pitch sliding (portamento), and low-pass vocal tract filtering.
    """
    audio_segments = []

    for note, dur in zip(midi_notes, note_durations):
        num_samples = int(sr * dur)
        t = np.linspace(0, dur, num_samples, endpoint=False)

        # 1. Base Frequency Target
        base_freq = midi_to_hz(note)

        # 2. Add Natural Vocal Vibrato (Pitch Oscillation)
        # depth in Hz = vibrato_depth semitones variation
        freq_variation = base_freq * (2 ** (vibrato_depth / 12.0) - 1)
        instantaneous_freq = base_freq + freq_variation * np.sin(2 * np.pi * vibrato_rate * t)

        # 3. Add Minor Pitch Drift/Unsteadiness
        drift = np.random.normal(0, 1.5, size=num_samples)
        b, a = signal.butter(2, 2.0 / (sr / 2), btype='low')
        smooth_drift = signal.lfilter(b, a, drift)
        instantaneous_freq += smooth_drift

        # Integrate frequency to get phase
        phase = 2 * np.pi * np.cumsum(instantaneous_freq) / sr

        # 4. Fundamental Wave + Subtle 2nd Harmonic
        raw_wave = np.sin(phase) + 0.25 * np.sin(2 * phase)

        # 5. Apply ADSR Envelope (Smooth transitions between notes)
        envelope = np.ones(num_samples)
        attack = int(0.05 * sr)   # 50ms attack
        release = int(0.05 * sr)  # 50ms release
        if num_samples > (attack + release):
            envelope[:attack] = np.linspace(0, 1, attack)
            envelope[-release:] = np.linspace(1, 0, release)

        audio_segments.append(raw_wave * envelope)

    # Concatenate sequence
    raw_audio = np.concatenate(audio_segments)

    # 6. Apply Low-Pass Filter to Simulate Vocal Tract Muffling (The "Hum")
    b_lp, a_lp = signal.butter(4, cutoff_freq / (sr / 2), btype='low')
    hummed_audio = signal.lfilter(b_lp, a_lp, raw_audio)

    # Normalize amplitude
    hummed_audio = hummed_audio / np.max(np.abs(hummed_audio))
    return hummed_audio, sr

# --- Example Usage ---
# C Major Arpeggio: C4 (60), E4 (64), G4 (67), C5 (72)
midi_sequence = [63, 54, 57, 62]
durations = [0.5, 0.5, 0.5, 1.0]  # seconds

audio, sample_rate = generate_synthetic_hum(midi_sequence, durations, vibrato_rate=5, vibrato_depth=2.0)

# Extract Spectrogram representation for Deep Learning input
stft = librosa.stft(audio)
spectrogram_db = librosa.amplitude_to_db(np.abs(stft), ref=np.max)

from IPython.display import Audio

# Play the array directly from memory (C Major Arpeggio hum)
Audio(audio, rate=sample_rate)